# Day 08: Enterprise Pipeline Engineering, Logging & Automated Testing
> **Part 8 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2008%20of%2010-008080?style=for-the-badge)](../README.md)

---

## ⚙️ Day 08: Enterprise Pipeline Engineering, Logging & Automated Testing
### 🎯 Learning Objectives:
1. Decouple pipeline parameters into an external centralized `CONFIG` management structure.
2. Implement enterprise logging with dual handlers (console streaming + persistent `pipeline.log` file).
3. Implement fail-safe exception handling with structured error logging.
4. Enforce strict **Data Quality SLAs** (e.g. automatically aborting pipeline if rejection rate exceeds 30%).
5. Generate an auditable execution report saved as `pipeline_report.json`.
6. Construct an automated **Unit Test Suite** with PySpark assertions:
   - Test 1: Record Reconciliation (`Total = Valid + Rejected`)
   - Test 2: Schema Cleanliness (Zero invalid customer IDs in Silver)
   - Test 3: Business Boundary Validation (Zero negative revenues)
   - Test 4: Financial Calculation Integrity (`Revenue == Quantity * Price`).
---


Logging + Configuration + Error Handling + Testing
The goal is:
                    Pipeline
                       │
             ┌─────────┼─────────┐
             ▼         ▼         ▼
          Config     Logging   Validation
             │         │         │
             └─────────┼─────────┘
                       ▼
                 ETL Processing
                       │
              ┌────────┴────────┐
              ▼                 ▼
           SUCCESS             ERROR
              │                 │
              ▼                 ▼
          Output Data       Error Log

Today you'll learn four things:
1. Configuration — don't hardcode paths everywhere.
2. Logging — know exactly what your pipeline did.
3. Exception handling — fail gracefully.
4. Testing — verify that your pipeline produces correct results.


### ⚙️ Initialize SparkSession for Day 08


In [254]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

from pathlib import Path
import logging
import json
from datetime import datetime

spark = (
    SparkSession.builder
    .appName("Day8_PipelineEngineering")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [255]:
CONFIG = {
    "project_name": "ecommerce_pipeline",

    "source_path": "day7_project/source/orders.csv",

    "bronze_path": "day7_project/bronze",

    "silver_path": "day7_project/silver",

    "gold_path": "day7_project/gold",

    "rejected_path": "day7_project/rejected",

    "log_path": "day7_project/logs"
}

CONFIG

{'project_name': 'ecommerce_pipeline',
 'source_path': 'day7_project/source/orders.csv',
 'bronze_path': 'day7_project/bronze',
 'silver_path': 'day7_project/silver',
 'gold_path': 'day7_project/gold',
 'rejected_path': 'day7_project/rejected',
 'log_path': 'day7_project/logs'}

In [256]:
for key in [
    "bronze_path",
    "silver_path",
    "gold_path",
    "rejected_path",
    "log_path"
]:
    Path(CONFIG[key]).mkdir(
        parents=True,
        exist_ok=True
    )

print("Directories ready")

Directories ready


In [257]:
from pathlib import Path
import logging

# 1. Ensure directory exists
log_dir = Path(CONFIG["log_path"])
log_dir.mkdir(parents=True, exist_ok=True)
log_file = log_dir / "pipeline.log"

# 2. Configure logger and clear existing handlers if re-running in a notebook
logger = logging.getLogger("EcommercePipeline")
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()

# 3. Add File Handler
file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

logger.info("Pipeline logger initialized")
print("Log file location:", log_file.resolve())

2026-10-08 07:43:35,761 - INFO - Pipeline logger initialized


Log file location: C:\Users\Me\Documents\Machine Learning\Pyspark\day7_project\logs\pipeline.log


In [258]:
logger.info("Starting Day 8 pipeline")
logger.warning("This is a test warning")
logger.error("This is a test error")

2026-10-08 07:43:35,790 - INFO - Starting Day 8 pipeline
2026-10-08 07:43:35,793 - WARNING - This is a test warning
2026-10-08 07:43:35,796 - ERROR - This is a test error


In [259]:
try:
    df = spark.read.csv(
        "does_not_exist.csv"
    )

    print(df.count())

except Exception as e:
    logger.error(
        f"Failed to read source file: {e}"
    )

    print("Pipeline failed while reading source")

2026-10-08 07:43:35,878 - ERROR - Failed to read source file: [PATH_NOT_FOUND] Path does not exist: file:/C:/Users/Me/Documents/Machine Learning/Pyspark/does_not_exist.csv. SQLSTATE: 42K03


Pipeline failed while reading source


In [260]:
def ingest_data(spark, config):

    source_path = config["source_path"]

    logger.info(
        f"Starting ingestion: {source_path}"
    )

    try:

        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(source_path)
        )

        record_count = df.count()

        logger.info(
            f"Ingestion successful. Records: {record_count}"
        )

        return df

    except Exception as e:

        logger.exception(
            f"Ingestion failed: {e}"
        )

        raise

catching the error
       ↓
logging it
       ↓
raising it again


In [262]:
raw_df = ingest_data(
    spark,
    CONFIG
)

raw_df.show()

2026-10-08 07:43:35,941 - INFO - Starting ingestion: day7_project/source/orders.csv
2026-10-08 07:43:36,291 - INFO - Ingestion successful. Records: 10


+--------+-----------+--------+--------+-----+----------+
|order_id|customer_id| product|quantity|price|order_date|
+--------+-----------+--------+--------+-----+----------+
|    1001|        101|  Laptop|       1|55000|2026-10-01|
|    1002|        102|   Mouse|       2|  800|2026-10-01|
|    1003|        103|Keyboard|       1| 1500|2026-10-02|
|    1004|        104| Monitor|       2|12000|2026-10-02|
|    1005|        105|  Laptop|      -1|55000|2026-10-03|
|    1006|        101|   Mouse|       3|  800|2026-10-03|
|    1007|       NULL|Keyboard|       1| 1500|2026-10-03|
|    1008|        106| Monitor|       2|12000|2026-10-04|
|    1008|        106| Monitor|       2|12000|2026-10-04|
|    1009|        107|  Laptop|       1|55000|   INVALID|
+--------+-----------+--------+--------+-----+----------+



In [263]:
from pyspark.sql import functions as F
import builtins

def validate_data(df):
    logger.info("Starting data validation")
    
    checked = (
        df
        .withColumn(
            "parsed_date", 
            F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd"))
        )
        .withColumn(
            "validation_errors",
            F.array_compact(
                F.array(
                    F.when(F.col("order_id").isNull(), F.lit("Missing order ID")),
                    F.when(F.col("customer_id").isNull(), F.lit("Missing customer ID")),
                    F.when(F.col("quantity").isNull() | (F.col("quantity") <= 0), F.lit("Invalid quantity")),
                    F.when(F.col("price").isNull() | (F.col("price") <= 0), F.lit("Invalid price")),
                    F.when(F.col("order_date").isNotNull() & F.col("parsed_date").isNull(), F.lit("Invalid date format"))
                )
            )
        )
    )
    
    # Separate valid rows (no errors) from rejected rows (has errors)
    valid_df = checked.filter(F.size("validation_errors") == 0)
    rejected_df = checked.filter(F.size("validation_errors") > 0)
    
    valid_count = valid_df.count()
    rejected_count = rejected_df.count()
    
    logger.info(
        f"Validation complete | "
        f"Valid: {valid_count} | "
        f"Rejected: {rejected_count}"
    )
    
    return valid_df, rejected_df


In [264]:
valid_df, rejected_df = validate_data(
    raw_df
)

valid_df.show()

2026-10-08 07:43:36,453 - INFO - Starting data validation
2026-10-08 07:43:36,841 - INFO - Validation complete | Valid: 7 | Rejected: 3


+--------+-----------+--------+--------+-----+----------+-------------------+-----------------+
|order_id|customer_id| product|quantity|price|order_date|        parsed_date|validation_errors|
+--------+-----------+--------+--------+-----+----------+-------------------+-----------------+
|    1001|        101|  Laptop|       1|55000|2026-10-01|2026-10-01 00:00:00|               []|
|    1002|        102|   Mouse|       2|  800|2026-10-01|2026-10-01 00:00:00|               []|
|    1003|        103|Keyboard|       1| 1500|2026-10-02|2026-10-02 00:00:00|               []|
|    1004|        104| Monitor|       2|12000|2026-10-02|2026-10-02 00:00:00|               []|
|    1006|        101|   Mouse|       3|  800|2026-10-03|2026-10-03 00:00:00|               []|
|    1008|        106| Monitor|       2|12000|2026-10-04|2026-10-04 00:00:00|               []|
|    1008|        106| Monitor|       2|12000|2026-10-04|2026-10-04 00:00:00|               []|
+--------+-----------+--------+--------+

In [265]:
rejected_df.select(
    "order_id",
    "validation_errors"
).show(truncate=False)

+--------+---------------------+
|order_id|validation_errors    |
+--------+---------------------+
|1005    |[Invalid quantity]   |
|1007    |[Missing customer ID]|
|1009    |[Invalid date format]|
+--------+---------------------+



In [266]:
total_records = raw_df.count()
valid_records = valid_df.count()
rejected_records = rejected_df.count()

rejection_rate = (
    rejected_records / total_records
    if total_records > 0
    else 0
)

logger.info(
    f"Pipeline metrics | "
    f"Total={total_records} | "
    f"Valid={valid_records} | "
    f"Rejected={rejected_records} | "
    f"RejectionRate={rejection_rate:.2%}"
)

print("Total:", total_records)
print("Valid:", valid_records)
print("Rejected:", rejected_records)
print("Rejection rate:", builtins.round(rejection_rate * 100, 2), "%")


2026-10-08 07:43:37,618 - INFO - Pipeline metrics | Total=10 | Valid=7 | Rejected=3 | RejectionRate=30.00%


Total: 10
Valid: 7
Rejected: 3
Rejection rate: 30.0 %


In [267]:
MAX_REJECTION_RATE = 0.30

if rejection_rate > MAX_REJECTION_RATE:

    logger.error(
        f"Rejection rate {rejection_rate:.2%} "
        f"exceeded allowed limit "
        f"{MAX_REJECTION_RATE:.2%}"
    )

    raise ValueError(
        "Data quality threshold exceeded"
    )

else:

    logger.info(
        "Data quality threshold passed"
    )

2026-10-08 07:43:37,638 - INFO - Data quality threshold passed


In [268]:
pipeline_report = {
    "pipeline": CONFIG["project_name"],
    "run_time": datetime.now().isoformat(),
    "records_received": total_records,
    "records_valid": valid_records,
    "records_rejected": rejected_records,
    "rejection_rate": builtins.round(
        rejection_rate,
        4
    ),
    "status": "SUCCESS"
}

pipeline_report


{'pipeline': 'ecommerce_pipeline',
 'run_time': '2026-10-08T07:43:37.660756',
 'records_received': 10,
 'records_valid': 7,
 'records_rejected': 3,
 'rejection_rate': 0.3,
 'status': 'SUCCESS'}

In [269]:
report_path = (
    Path(CONFIG["log_path"])
    / "pipeline_report.json"
)

with open(
    report_path,
    "w"
) as f:

    json.dump(
        pipeline_report,
        f,
        indent=4
    )

print(
    f"Report saved to {report_path}"
)

Report saved to day7_project\logs\pipeline_report.json


day7_project/
│
├── logs/
│   ├── pipeline.log
│   └── pipeline_report.json


In [271]:
assert total_records == (
    valid_records + rejected_records
)

print("✅ Record reconciliation passed")

✅ Record reconciliation passed


In [272]:
assert rejected_records > 0

print("✅ Invalid-record detection passed")

✅ Invalid-record detection passed


In [273]:
transformed_df = (
    valid_df
    .withColumn(
        "order_date",
        F.col("parsed_date")
    )
    .withColumn(
        "revenue",
        F.col("quantity") * F.col("price")
    )
)

In [274]:
negative_revenue = transformed_df.filter(
    F.col("revenue") < 0
).count()

assert negative_revenue == 0

print("✅ Revenue validation passed")

✅ Revenue validation passed


### 🚀 Master Pipeline Driver Function: `run_pipeline`


In [275]:
def run_pipeline(spark, config):

    start_time = datetime.now()

    try:

        logger.info("========== PIPELINE START ==========")

        raw_df = ingest_data(spark, config)
        valid_df, rejected_df = validate_data(raw_df)

        total_records = raw_df.count()
        valid_records = valid_df.count()
        rejected_records = rejected_df.count()
        rejection_rate = rejected_records / total_records if total_records > 0 else 0.0

        assert total_records == (valid_records + rejected_records)

        MAX_REJECTION_RATE = 0.30
        if rejection_rate > MAX_REJECTION_RATE:
            logger.error(f"Rejection rate {rejection_rate:.2%} exceeded limit {MAX_REJECTION_RATE:.2%}")
            raise ValueError("Data quality threshold exceeded")

        transformed_df = (
            valid_df
            .withColumn("order_date", F.col("parsed_date"))
            .withColumn("revenue", F.col("quantity") * F.col("price"))
        )

        report_path = Path(config["log_path"]) / "pipeline_report.json"
        pipeline_report = {
            "pipeline": config["project_name"],
            "run_time": datetime.now().isoformat(),
            "records_received": total_records,
            "records_valid": valid_records,
            "records_rejected": rejected_records,
            "rejection_rate": builtins.round(rejection_rate, 4),
            "status": "SUCCESS"
        }

        with open(report_path, "w") as f:
            json.dump(pipeline_report, f, indent=4)

        logger.info("========== PIPELINE SUCCESS ==========")
        return pipeline_report

    except Exception as e:

        logger.exception(
            f"========== PIPELINE FAILED: {e} =========="
        )

        raise


### ⚙️ Initialize SparkSession for Day 08


In [276]:
import os
import json
import logging
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Day8_Production_Pipeline")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark started successfully")

Spark started successfully


In [277]:
CONFIG = {
    "source_path": "day7_project/source/orders.csv",
    "bronze_path": "day7_project/output/bronze/orders",
    "silver_path": "day7_project/output/silver/orders",
    "gold_path": "day7_project/output/gold/customer_report",
    "rejected_path": "day7_project/output/rejected/orders",
    "log_path": "day7_project/logs/pipeline.log",
    "report_path": "day7_project/output/pipeline_report.json"
}

In [278]:
for key, path in CONFIG.items():
    if key.endswith("_path"):
        if "." not in os.path.basename(path):
            os.makedirs(path, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(path), exist_ok=True)
print("Directories ready")

Directories ready


In [279]:
logging.basicConfig(
    filename=CONFIG["log_path"],
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Day8Pipeline")
logger.info("Pipeline logging initialized")
print("Logging configured")

2026-10-08 07:43:38,091 - INFO - Pipeline logging initialized


Logging configured


In [280]:
def ingest_data(spark, config):
    try:
        logger.info("Starting data ingestion")
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(config["source_path"])
        )
        logger.info("Data ingestion successful")
        return df
    except Exception as e:
        logger.exception("Data ingestion failed")
        raise

In [281]:
def validate_data(df):
    logger.info("Starting data validation")
    validated = (
        df
        .withColumn(
            "parsed_order_date",
            F.try_to_date(F.col("order_date"), "yyyy-MM-dd")  # <-- Changed from to_date to try_to_date
        )
        .withColumn(
            "validation_error",
            F.concat_ws(
                ", ",
                F.when(F.col("order_id").isNull(), "missing_order_id"),
                F.when(
                    (F.col("customer_id").isNull()) | (F.col("customer_id") <= 0),
                    "invalid_customer_id"
                ),
                F.when(
                    (F.col("quantity").isNull()) | (F.col("quantity") <= 0),
                    "invalid_quantity"
                ),
                F.when(
                    (F.col("price").isNull()) | (F.col("price") < 100),
                    "invalid_price"
                ),
                F.when(
                    F.col("parsed_order_date").isNull(),
                    "invalid_order_date"
                )
            )
        )
    )
    
    valid_df = validated.filter(F.col("validation_error") == "")
    rejected_df = validated.filter(F.col("validation_error") != "")
    
    logger.info("Validation completed")
    return valid_df, rejected_df

In [282]:
import builtins  # Explicitly use Python's built-in round

def calculate_metrics(df, valid_df, rejected_df):
    total_count = df.count()
    valid_count = valid_df.count()
    rejected_count = rejected_df.count()
    
    rejection_rate = (
        (rejected_count / total_count * 100)
        if total_count > 0
        else 0
    )
    
    metrics = {
        "total_records": total_count,
        "valid_records": valid_count,
        "rejected_records": rejected_count,
        "rejection_rate": builtins.round(rejection_rate, 2)  # Use builtins.round
    }
    
    logger.info(f"Metrics: {metrics}")
    return metrics

### 🚀 Master Pipeline Driver Function: `run_pipeline`


In [283]:
import builtins
import json
from datetime import datetime

def run_pipeline(spark, config):
    try:
        # 1. Log pipeline start
        logger.info("=== PIPELINE RUN STARTED ===")
        
        # 2. Ingest data
        df = ingest_data(spark, config)
        
        # 3. Validate data
        valid_df, rejected_df = validate_data(df)
        
        # 4. Calculate metrics
        metrics = calculate_metrics(df, valid_df, rejected_df)
        
        # 5. Check rejection threshold (e.g., max 30%)
        MAX_REJECTION_RATE = 30.0
        if metrics["rejection_rate"] > MAX_REJECTION_RATE:
            logger.error(
                f"Rejection rate {metrics['rejection_rate']}% "
                f"exceeded threshold {MAX_REJECTION_RATE}%"
            )
            raise ValueError("Data quality threshold exceeded. Pipeline stopped.")
        else:
            logger.info(
                f"Data quality passed: {metrics['rejection_rate']}% rejection rate"
            )
        
        # 6. Transform valid data
        logger.info("Starting transformations")
        transformed_df = (
            valid_df
            .withColumn("revenue", F.col("quantity") * F.col("price"))
            .withColumn("order_year", F.year("parsed_order_date"))
            .withColumn("order_month", F.month("parsed_order_date"))
        )
        
        # 7. Write Silver Data
        (
            transformed_df
            .write
            .mode("overwrite")
            .parquet(config["silver_path"])
        )
        logger.info("Silver data written successfully")
        
        # 8. Write Rejected Data
        (
            rejected_df
            .write
            .mode("overwrite")
            .parquet(config["rejected_path"])
        )
        logger.info("Rejected records written successfully")
        
        # 9. Create Gold Report (Customer-level aggregates)
        customer_report = (
            transformed_df
            .groupBy("customer_id")
            .agg(
                F.count("*").alias("total_orders"),
                F.sum("revenue").alias("total_revenue"),
                F.avg("revenue").alias("avg_order_value")
            )
            .orderBy(F.col("total_revenue").desc())
        )
        
        (
            customer_report
            .write
            .mode("overwrite")
            .parquet(config["gold_path"])
        )
        logger.info("Gold customer report written successfully")
        
        # 10. Create Pipeline JSON Report
        pipeline_report = {
            "pipeline": "Day8_Production_Pipeline",
            "run_time": datetime.now().isoformat(),
            "status": "SUCCESS",
            "metrics": metrics,
            "outputs": {
                "silver": config["silver_path"],
                "gold": config["gold_path"],
                "rejected": config["rejected_path"]
            }
        }
        
        with open(config["report_path"], "w") as f:
            json.dump(pipeline_report, f, indent=4)
        
        logger.info("Pipeline report generated successfully")
        logger.info("=== PIPELINE RUN COMPLETED SUCCESSFULLY ===")
        print("Pipeline executed successfully!")
        
        return transformed_df, metrics

    except Exception as e:
        logger.exception("Pipeline failed with exception")
        print(f"Pipeline failed: {e}")
        raise

# Execute the pipeline
transformed_df, metrics = run_pipeline(spark, CONFIG)

2026-10-08 07:43:38,188 - INFO - === PIPELINE RUN STARTED ===
2026-10-08 07:43:38,188 - INFO - Starting data ingestion
2026-10-08 07:43:38,368 - INFO - Data ingestion successful
2026-10-08 07:43:38,368 - INFO - Starting data validation
2026-10-08 07:43:38,477 - INFO - Validation completed
2026-10-08 07:43:38,858 - INFO - Metrics: {'total_records': 10, 'valid_records': 7, 'rejected_records': 3, 'rejection_rate': 30.0}
2026-10-08 07:43:38,867 - INFO - Data quality passed: 30.0% rejection rate
2026-10-08 07:43:38,867 - INFO - Starting transformations
2026-10-08 07:43:39,608 - INFO - Silver data written successfully
2026-10-08 07:43:40,338 - INFO - Rejected records written successfully
2026-10-08 07:43:41,148 - INFO - Gold customer report written successfully
2026-10-08 07:43:41,148 - INFO - Pipeline report generated successfully
2026-10-08 07:43:41,156 - INFO - === PIPELINE RUN COMPLETED SUCCESSFULLY ===


Pipeline executed successfully!


transformed_df, metrics = run_pipeline(spark, CONFIG)


### 🧪 Automated Production Unit Test Suite


In [285]:
print("\nRunning pipeline unit tests...")

# Test 1 — Reconciliation
assert metrics["total_records"] == metrics["valid_records"] + metrics["rejected_records"], \
    "Test 1 Failed: Record count reconciliation mismatch!"
print("Test 1 PASSED: Reconciliation matches")

# Test 2 — No invalid customer IDs in Silver layer
invalid_customers = transformed_df.filter(F.col("customer_id") <= 0).count()
assert invalid_customers == 0, "Test 2 Failed: Found invalid customer IDs in silver!"
print("Test 2 PASSED: No invalid customer IDs")

# Test 3 — No negative revenue
negative_revenue = transformed_df.filter(F.col("revenue") < 0).count()
assert negative_revenue == 0, "Test 3 Failed: Found negative revenue!"
print("Test 3 PASSED: No negative revenue")

# Test 4 — Revenue calculation check
revenue_errors = transformed_df.filter(
    F.col("revenue") != F.col("quantity") * F.col("price")
).count()
assert revenue_errors == 0, "Test 4 Failed: Revenue calculation error!"
print("Test 4 PASSED: Revenue calculations accurate")

print("\nAll tests passed successfully! Ready for Day 9.")


Running pipeline unit tests...
Test 1 PASSED: Reconciliation matches
Test 2 PASSED: No invalid customer IDs
Test 3 PASSED: No negative revenue
Test 4 PASSED: Revenue calculations accurate

All tests passed successfully! Ready for Day 9.


---
### 🎓 Day 08 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 09: Cloud Database Connectivity & Neon PostgreSQL JDBC Lakehouse Ingestion**](./Day_09_Cloud_PostgreSQL_JDBC_and_Lakehouse_Ingestion.ipynb)
